<a href="https://colab.research.google.com/github/michaelsteven1299/proyecto_michael-/blob/main/src/02_limpieza.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Universidad Libre - Seccional Cali<br>Facultad de Ingeniería - Diplomado en Ciencia de Datos<br>(ↄ) Diego Fernando Marin, 2024

# 02_limpieza
Plantilla para el desarrollo del proyecto del diplomado de Ciencia de Datos, aplicando buenas prácticas.

---

Este cuaderno se dedica a la crucial tarea de preparar los datos para el análisis, asegurando su calidad y consistencia. La limpieza de datos es fundamental para obtener resultados confiables en las etapas posteriores.

**Propósito:** Mejorar la calidad de los datos mediante la corrección de errores, estandarización de formatos y aplicación de reglas de calidad.

**Tareas habituales:**
- Detección y tratamiento de valores nulos (Completar datos faltantes)
- Estandarización de formatos (nombres, fechas, números, texto)
- Identificación y corrección de outliers
- Corrección de errores tipográficos
- Normalizar/Homogeneizar datos
- Validación de reglas de negocio
- Documentación de transformaciones aplicadas
- Verificación de consistencia en los datos
- Eliminar columnas o filas
- Crear columnas nuevas

IMPORTAMOS LOS DATOS Y LOS ANALIZAMOS SI TENEMOS NULOS ETC

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

LANDING_PATH = "/content/drive/MyDrive/proyecto_oro/data/landing/"
df = pd.read_csv(LANDING_PATH + "consolidado_oro_dxy.csv", index_col='DATE', parse_dates=True)

print(f"Filas: {len(df):,} | Columnas: {len(df.columns)}")
print(f"Rango: {df.index.min().date()} → {df.index.max().date()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Filas: 1,431 | Columnas: 32
Rango: 2021-01-01 → 2026-07-01


In [5]:
nulos = df.isna().sum().sort_values(ascending=False)
print(nulos)

oro_cop              419
trm_trm              404
oro_xauusd_Close      53
oro_xauusd_High       53
bono_10y_Volume       53
bono_10y_Open         53
bono_10y_Close        53
oro_xauusd_Volume     53
oro_xauusd_Low        53
oro_xauusd_Open       53
bono_10y_Low          53
bono_10y_High         53
vix_Close             52
vix_Open              52
vix_Low               52
vix_High              52
vix_Volume            52
dxy_Open              51
wti_crudo_Low         51
wti_crudo_High        51
dxy_High              51
wti_crudo_Close       51
wti_crudo_Volume      51
dxy_Volume            51
dxy_Close             51
dxy_Low               51
wti_crudo_Open        51
usd_cop_Volume         3
usd_cop_Close          3
usd_cop_Open           3
usd_cop_High           3
usd_cop_Low            3
dtype: int64


In [6]:
# Recalculamos oro_cop usando usd_cop (mercado) en vez de la TRM oficial,
# porque usd_cop tiene una cobertura de datos mucho más completa
# (solo 3 nulos vs. 404 de la TRM), y la diferencia entre ambas fuentes
# es mínima para un modelo de retornos porcentuales.
df['oro_cop'] = df['oro_xauusd_Close'] * df['usd_cop_Close']

# Ahora hacemos dropna solo sobre las columnas relevantes para el modelo
# (dejamos trm_trm fuera del criterio, ya que no la usaremos como
# variable principal, pero la conservamos en el dataframe como contexto)
columnas_criticas = [col for col in df.columns if col != 'trm_trm']

filas_antes = len(df)
df_limpio = df.dropna(subset=columnas_criticas)
filas_despues = len(df_limpio)

print(f"Filas antes: {filas_antes:,}")
print(f"Filas después: {filas_despues:,}")
print(f"Descartadas: {filas_antes - filas_despues:,}")

Filas antes: 1,431
Filas después: 1,377
Descartadas: 54


In [7]:
print(df_limpio[columnas_criticas].isna().sum().sum())  # debería dar 0
print(f"\nRango final: {df_limpio.index.min()} → {df_limpio.index.max()}")

0

Rango final: 2021-01-04 00:00:00 → 2026-06-30 00:00:00


In [8]:
CLEAN_PATH = "/content/drive/MyDrive/proyecto_oro/data/clean/"
import os
os.makedirs(CLEAN_PATH, exist_ok=True)

df_limpio.to_csv(CLEAN_PATH + "oro_cop_limpio.csv")
print(f"✓ Guardado: {len(df_limpio):,} filas × {len(df_limpio.columns)} columnas")

✓ Guardado: 1,377 filas × 32 columnas


In [9]:
# Verificación de tipos de datos
print(df_limpio.dtypes)

# Verificación de duplicados en el índice de fechas
print(f"\nFechas duplicadas: {df_limpio.index.duplicated().sum()}")

# Verificación de valores negativos en columnas de precio/volumen
columnas_precio = [c for c in df_limpio.columns if 'Close' in c or 'Open' in c or 'High' in c or 'Low' in c]
for col in columnas_precio:
    negativos = (df_limpio[col] < 0).sum()
    if negativos > 0:
        print(f"⚠ {col}: {negativos} valores negativos")

dxy_Close            float64
dxy_High             float64
dxy_Low              float64
dxy_Open             float64
dxy_Volume           float64
usd_cop_Close        float64
usd_cop_High         float64
usd_cop_Low          float64
usd_cop_Open         float64
usd_cop_Volume       float64
wti_crudo_Close      float64
wti_crudo_High       float64
wti_crudo_Low        float64
wti_crudo_Open       float64
wti_crudo_Volume     float64
vix_Close            float64
vix_High             float64
vix_Low              float64
vix_Open             float64
vix_Volume           float64
oro_xauusd_Close     float64
oro_xauusd_High      float64
oro_xauusd_Low       float64
oro_xauusd_Open      float64
oro_xauusd_Volume    float64
trm_trm              float64
bono_10y_Close       float64
bono_10y_High        float64
bono_10y_Low         float64
bono_10y_Open        float64
bono_10y_Volume      float64
oro_cop              float64
dtype: object

Fechas duplicadas: 0
